# Задание Lite к занятию Методы Faiss, Langchain

Возьмите базу знаний "Правила безопасности опасных производственных объектов"

URL: https://docs.google.com/document/d/1q4l912Re8zuIfBax4FDS3ZppYmVPzER3Si2wrmznddc

или любой другой текстовый документ по ссылке с ГуглДиска.

- Разбейте текст на чанки с использованием CharacterTextSplitter
- Создайте векторную базу и сохраниете ее на ГуглДиск
- Загрузите из ГуглДиска сохраненную на предыдущем шаге векторную базу
- Создайте простого консультанта по базе знаний. Допускается использование любой большой языковой модели.
- В этом задании не используйте ретриверы и ChatOpenAI из Langchain.
- Задайте вопрос консультанту по базе знаний и получите ответ

# Решение

In [ ]:
%pip -q install faiss-cpu langchain-community langchain-text-splitters langchain-openai openai gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
%pip install -q "requests<2.32.4"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.3 which is incompatible.
google-adk 2.7.1 requires requests<3,>=2.32.4, but you have requests 2.32.3 which is incompatible.


In [ ]:
%pip install -q "requests==2.32.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [ ]:
%pip uninstall -y langchain-community langchain-openai

Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-openai 1.7.2
Uninstalling langchain-openai-1.7.2:
  Successfully uninstalled langchain-openai-1.7.2


In [5]:
import requests
import faiss
from langchain_text_splitters import CharacterTextSplitter
from openai import OpenAI

print("requests:", requests.__version__)
print("FAISS, CharacterTextSplitter и OpenAI импортированы")

requests: 2.32.4
FAISS, CharacterTextSplitter и OpenAI импортированы


In [7]:
from google.colab import userdata

try:
    api_key = userdata.get("OPENAI_API_KEY")
    print("Ключ OpenAI найден")
except Exception:
    print("Ключ OpenAI не найден в секретах Colab")

Ключ OpenAI найден


In [8]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
import requests

doc_id = "1q4l912Re8zuIfBax4FDS3ZppYmVPzER3Si2wrmznddc"
url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"

response = requests.get(url)
response.raise_for_status()
document_text = response.text

print("Символов загружено:", len(document_text))
print(document_text[:500])


Символов загружено: 239830
﻿"Правила безопасности опасных производственных объектов, на которых используются подъемные сооружения"
 I. Общие положения 1. Настоящие Федеральные нормы и правила в области промышленной безопасности "Правила безопасности опасных производственных объектов, на которых используются подъемные сооружения" (далее - ФНП) разработаны в соответствии с Федеральным законом от 21 июля 1997 г. N 116-ФЗ "О промышленной безопасности опасных производственных объектов" (Собрание законодательства Российской Фе


In [10]:
splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=1200,
    chunk_overlap=200,
    length_function=len
)

chunks = splitter.split_text(document_text)

print("Фрагментов создано:", len(chunks))
print("Первый фрагмент:\n", chunks[0][:700])


Фрагментов создано: 240
Первый фрагмент:
 ﻿"Правила безопасности опасных производственных объектов, на которых используются подъемные сооружения"
 I. Общие положения 1. Настоящие Федеральные нормы и правила в области промышленной безопасности "Правила безопасности опасных производственных объектов, на которых используются подъемные сооружения" (далее - ФНП) разработаны в соответствии с Федеральным законом от 21 июля 1997 г. N 116-ФЗ "О промышленной безопасности опасных производственных объектов" (Собрание законодательства Российской Федерации, 1997, N 30, ст.3588; 2018, N 31, ст.4860) (далее - Федеральный закон N 116-ФЗ). ФНП устанавливают необходимые требования к деятельности в области промышленной безопасности на опасных производ


In [11]:
import os
import json
import numpy as np
import faiss

client = OpenAI(api_key=api_key)

vectors = []
batch_size = 64

for start in range(0, len(chunks), batch_size):
    batch = chunks[start:start + batch_size]
    result = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    vectors.extend(item.embedding for item in result.data)

vectors = np.array(vectors, dtype="float32")
faiss.normalize_L2(vectors)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

save_dir = "/content/drive/MyDrive/Урок 12 — Продвинутые методы работы с Faiss/faiss_db"
os.makedirs(save_dir, exist_ok=True)

faiss.write_index(index, f"{save_dir}/index.faiss")

with open(f"{save_dir}/chunks.json", "w", encoding="utf-8") as file:
    json.dump(chunks, file, ensure_ascii=False)

print("Векторов в индексе:", index.ntotal)
print("Размерность вектора:", vectors.shape[1])
print("Сохранено в Google Drive:", save_dir)

Векторов в индексе: 240
Размерность вектора: 1536
Сохранено в Google Drive: /content/drive/MyDrive/Урок 12 — Продвинутые методы работы с Faiss/faiss_db


In [12]:
import json
import faiss

index_loaded = faiss.read_index(f"{save_dir}/index.faiss")

with open(f"{save_dir}/chunks.json", "r", encoding="utf-8") as file:
    chunks_loaded = json.load(file)

print("Загружено векторов:", index_loaded.ntotal)
print("Загружено фрагментов:", len(chunks_loaded))

Загружено векторов: 240
Загружено фрагментов: 240


In [13]:
question = "Какие обязанности есть у специалиста, ответственного за безопасное производство работ с применением подъемных сооружений?"

query_result = client.embeddings.create(
    model="text-embedding-3-small",
    input=question
)

query_vector = np.array(
    [query_result.data[0].embedding],
    dtype="float32"
)
faiss.normalize_L2(query_vector)

scores, ids = index_loaded.search(query_vector, k=4)

for rank, chunk_id in enumerate(ids[0], start=1):
    print(f"\nФрагмент {rank} | сходство: {scores[0][rank - 1]:.3f}")
    print(chunks_loaded[chunk_id][:1000])


Фрагмент 1 | сходство: 0.618
их монтаж. Монтаж ограничителей, указателей и регистраторов параметров на уже изготовленных и находящихся в Об утверждении федеральных норм и правил в области промышленной безопасности "Правила безопасности опасных производственных объектов, на которых используются подъемные сооружения" Приказ Ростехнадзора от 26.11.2020 N 461 ФНП в области промышленной безопасности от 26.11.2020 N 461 Страница 9 Интранет эксплуатации ПС должен выполняться работниками специализированных организаций в соответствии с требованиями, установленными разработчиками или изготовителями указанных приборов, а также с учетом конструктивных особенностей и назначения ПС. Организация и планирование работ 29. Организации и их работники, выполняющие работы по монтажу (демонтажу), наладке, должны соответствовать требованиям, изложенным в пунктах 10-21 настоящих ФНП. 30. Перед выполнением работ все работники, выполняющие работы по монтажу (демонтажу) и наладке, должны быть ознакомлены с рабо

In [14]:
context = "\n\n".join(
    chunks_loaded[int(chunk_id)]
    for chunk_id in ids[0]
)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "Ты консультант по загруженному документу. "
                "Отвечай только на основании переданных выдержек. "
                "Если данных недостаточно, прямо скажи об этом."
            )
        },
        {
            "role": "user",
            "content": f"Выдержки из документа:\n{context}\n\nВопрос: {question}"
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

На основании переданных выдержек из документа, обязанности специалиста, ответственного за безопасное производство работ с применением подъемных сооружений (ПС), включают:

1. Ознакомление с рабочими процедурами и характеристикой выполняемых работ, знание должностных и производственных инструкций (пункт 30).

2. Ознакомление с руководством (инструкцией) по монтажу, демонтажу и наладке ПС (пункт 31).

3. Знание и умение выявлять дефекты и повреждения металлических конструкций, механизмов, электро-, пневмо-, гидрооборудования, систем управления ПС и приборов безопасности (ограничителей, указателей, регистраторов).

4. Умение выполнять наладочные работы на ПС, заявленных специализированной организацией (пункт г).

5. Применение технологий ремонта и восстановления узлов и деталей ПС, электро- и гидрооборудования, а также ограничителей, указателей, регистраторов и систем управления ПС (пункт д).

6. Умение применять такелажные и монтажные приспособления, грузоподъемные механизмы, стропы, соо